# Day 4 · 실습 1. YOLO26 객체 탐지와 Threshold 비교

**연계 이론:** Day 4 · 이론 2. Object Detection과 YOLO의 원리  
**권장 운영:** 50분 / 모델 실행 목표 10분 이내(설치·가중치 다운로드 시간 제외)  
**실습 유형:** B(조건 변경·비교) + C(시각화·관찰) / 학습(Training) 없음

### 학습 목표
1. 사전학습된 YOLO26 Nano로 이미지 속 객체의 클래스·Bounding Box·Confidence를 확인합니다.
2. **Confidence Threshold**만 변경하여 탐지 결과의 차이를 비교합니다.
3. 별도의 작은 예제로 IoU와 NMS의 중복 제거 원리를 확인합니다.

> **실행 순서:** 환경 확인 → 모델 로드 → Threshold 비교 → NMS 예제 → 정리. 제공된 셀을 위에서 아래로 실행하세요. 이 Notebook에서는 학습·평가용 Ground Truth가 없으므로 탐지 개수 변화만으로 정밀도/재현율 향상을 판단하지 않습니다.

## 0. 환경과 이미지 확인

GPU는 강의 환경의 RTX 4090을 전제로 합니다. `torch.cuda.is_available()`가 `False`면 실습은 CPU에서 실행되지만 속도가 달라질 수 있습니다. **처음 한 번만** 강사의 안내에 따라 터미널에서 `python -m pip install -r requirements.txt`를 실행하세요. 기존 PyTorch/CUDA는 담당자가 먼저 확인해야 합니다.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ZIP 안의 assets 폴더를 기준으로 작업 경로를 찾습니다.
def locate_root():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        for candidate in (parent, parent / "cv_day4_practice_v1"):
            if (candidate / "assets" / "coffee.jpg").is_file():
                return candidate
    raise FileNotFoundError(
        "assets/coffee.jpg를 찾지 못했습니다. ZIP 전체를 압축 해제하고 "
        "cv_day4_practice_v1 폴더에서 Notebook을 실행하세요."
    )

ROOT = locate_root()
ASSET_DIR = ROOT / "assets"
OUTPUT_DIR = ROOT / "outputs"
WEIGHT_DIR = ROOT / "weights"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("실습 루트:", ROOT)
print("샘플 이미지:", [p.name for p in sorted(ASSET_DIR.glob("*.jpg"))])

In [ ]:
import torch
try:
    import ultralytics
    from ultralytics import YOLO
except ImportError as exc:
    raise ImportError("ultralytics가 없습니다. 노트북의 Python 환경에서 %pip install -r requirements.txt 실행 후 커널을 재시작하세요.") from exc

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 모델 파일은 weights/에 사전 배치할 수 있습니다. 없으면 최초 실행 시 자동 다운로드됩니다.
MODEL_NAME = "yolo26n.pt"
MODEL_SOURCE = str(WEIGHT_DIR / MODEL_NAME) if (WEIGHT_DIR / MODEL_NAME).is_file() else MODEL_NAME
try:
    model = YOLO(MODEL_SOURCE)
except Exception as exc:
    raise RuntimeError("YOLO26 가중치를 불러오지 못했습니다. 인터넷/가중치 경로와 ultralytics 설치를 확인하세요.") from exc
print("모델:", MODEL_SOURCE, "| 모델 작업:", model.task)

In [ ]:
# 동일한 샘플로 조건을 비교합니다. 학생이 직접 임계값을 바꿔 볼 수 있습니다.
IMAGE_FILES = ["coffee.jpg", "chelsea.jpg"]
CONF_THRESHOLDS = [0.15, 0.30, 0.60]   # [조건 변경] 값을 바꿀 때는 같은 이미지·모델을 유지합니다.
IMGSZ = 640
for name in IMAGE_FILES:
    with Image.open(ASSET_DIR / name) as im:
        print(name, "원본 크기:", im.size)
assert len(CONF_THRESHOLDS) == len(set(CONF_THRESHOLDS))
assert all(0 <= t <= 1 for t in CONF_THRESHOLDS)

## 1. YOLO26 추론 및 Threshold별 결과 비교

`Results.boxes.xyxy`, `.conf`, `.cls`를 통해 객체 좌표·점수·클래스를 확인합니다. `result.plot()`은 주석이 표시된 이미지(BGR 배열)를 반환합니다. 아래에서는 RGB로 바꿔 비교합니다.

**관찰:** 같은 이미지에서 임계값을 올릴 때 표시되는 객체가 어떻게 바뀌나요? 이는 **후보 선택**의 변화이며 정답률 측정은 아닙니다.

In [ ]:
import csv

YOLO_OUT = OUTPUT_DIR / "yolo"
YOLO_OUT.mkdir(parents=True, exist_ok=True)

summary = {
    "model": MODEL_NAME,
    "model_source": MODEL_SOURCE,
    "device": str(DEVICE),
    "confidence_thresholds": CONF_THRESHOLDS,
    "images": {},
    "note": "Predictions only. Ground-truth evaluation is not performed here.",
}
count_rows = []

for image_name in IMAGE_FILES:
    image_path = ASSET_DIR / image_name
    with Image.open(image_path) as im:
        width, height = im.size
    record = {"width": width, "height": height, "thresholds": {}}
    fig, axes = plt.subplots(1, len(CONF_THRESHOLDS), figsize=(5 * len(CONF_THRESHOLDS), 5))
    axes = np.atleast_1d(axes)

    for ax, conf in zip(axes, CONF_THRESHOLDS):
        try:
            result = model.predict(
                source=str(image_path), conf=conf, imgsz=IMGSZ,
                device=DEVICE, save=False, verbose=False
            )[0]
        except Exception as exc:
            raise RuntimeError(f"{image_name} 추론 실패. 모델 파일과 GPU 메모리를 확인하세요.") from exc

        predictions = []
        if result.boxes is not None:
            xyxy = result.boxes.xyxy.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)
            for coords, score, class_id in zip(xyxy, scores, classes):
                predictions.append({
                    "class_id": int(class_id),
                    "class_name": str(result.names[int(class_id)]),
                    "confidence": float(score),
                    "xyxy": [float(v) for v in coords],
                })

        key = f"{conf:.2f}"
        record["thresholds"][key] = predictions
        count_rows.append({"image": image_name, "conf_threshold": key, "num_detections": len(predictions)})
        annotated_bgr = result.plot()
        annotated_rgb = annotated_bgr[:, :, ::-1]
        Image.fromarray(annotated_rgb).save(YOLO_OUT / f"{image_path.stem}_conf_{key}.png")
        ax.imshow(annotated_rgb)
        ax.set_title(f"conf ≥ {key} | {len(predictions)} detections")
        ax.axis("off")

    summary["images"][image_name] = record
    fig.suptitle(f"YOLO26 Nano — {image_name}")
    fig.tight_layout()
    fig.savefig(YOLO_OUT / f"{image_path.stem}_threshold_comparison.png", dpi=130, bbox_inches="tight")
    plt.show()

with (YOLO_OUT / "yolo_detections.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
with (YOLO_OUT / "threshold_counts.csv").open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "conf_threshold", "num_detections"])
    writer.writeheader()
    writer.writerows(count_rows)

print("저장:", YOLO_OUT)
for row in count_rows:
    print(row)

## 2. IoU와 NMS를 눈으로 확인하기

다음은 **실제 YOLO 모델의 내부 후보를 읽는 코드가 아니라**, 이론 2에서 학습한 세 개 Box를 이용하는 **설명용 계산 예제**입니다. A와 B가 많이 겹칠 때 B가 제거되는지 확인하고 IoU Threshold만 변경합니다.

In [ ]:
from matplotlib.patches import Rectangle

def box_iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - intersection
    return float(intersection / union) if union > 0 else 0.0

def greedy_nms(boxes, scores, iou_threshold=0.5):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        chosen = order.pop(0)
        keep.append(chosen)
        order = [j for j in order if box_iou_xyxy(boxes[chosen], boxes[j]) <= iou_threshold]
    return keep

BOXES = [[100, 100, 300, 300], [140, 100, 340, 300], [380, 100, 530, 300]]
SCORES = [0.92, 0.83, 0.70]
LABELS = ["A", "B", "C"]
print("IoU(A, B):", round(box_iou_xyxy(BOXES[0], BOXES[1]), 3))
assert abs(box_iou_xyxy(BOXES[0], BOXES[1]) - 2/3) < 1e-6

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (caption, keep) in zip(axes, [
    ("All candidates", [0, 1, 2]),
    ("NMS IoU = 0.50", greedy_nms(BOXES, SCORES, 0.50)),
    ("NMS IoU = 0.75", greedy_nms(BOXES, SCORES, 0.75)),
]):
    ax.set_xlim(0, 600)
    ax.set_ylim(400, 0)
    ax.set_aspect("equal")
    ax.set_title(caption)
    ax.grid(alpha=.12)
    for i in keep:
        x1, y1, x2, y2 = BOXES[i]
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2))
        ax.text(x1 + 5, y1 - 8, f"{LABELS[i]} {SCORES[i]:.2f}", fontsize=10)
    print(caption, "->", [LABELS[i] for i in keep])
fig.tight_layout()
fig.savefig(YOLO_OUT / "nms_illustration.png", dpi=140, bbox_inches="tight")
plt.show()

## 3. 선택 확인: YOLO26의 NMS-free 경로

YOLO26은 기본 one-to-many + NMS 경로와 `nms=False`의 one-to-one 경로를 제공합니다. 이 비교는 두 Head가 다르므로 **NMS만 껐다 켠 실험**으로 해석하면 안 됩니다. 공식 지원 버전에서만 실행되며, 실습의 필수 목표는 1·2절입니다.

In [ ]:
# 선택 실습: 수업 진도가 허용할 때만 확인합니다.
RUN_NMS_FREE_COMPARISON = False   # True로 바꾸면 실행
if RUN_NMS_FREE_COMPARISON:
    try:
        image_path = ASSET_DIR / IMAGE_FILES[0]
        result_nms_free = model.predict(
            source=str(image_path), conf=0.30, imgsz=IMGSZ,
            device=DEVICE, nms=False, save=False, verbose=False
        )[0]
        print("NMS-free 예측 객체 수:", len(result_nms_free.boxes))
        plt.figure(figsize=(8, 6))
        plt.imshow(result_nms_free.plot()[:, :, ::-1]); plt.axis("off"); plt.show()
    except Exception as exc:
        print("NMS-free 경로 미지원/실행 실패:", exc)
else:
    print("선택 실습을 건너뜁니다. RUN_NMS_FREE_COMPARISON=True로 바꾸어 실행할 수 있습니다.")

## 4. 핵심 질문 및 다음 실습 연결

1. Confidence Threshold 0.15·0.30·0.60에서 탐지 객체 수와 표시된 Box는 어떻게 달라졌나요?
2. Confidence Threshold와 **NMS IoU Threshold**는 각각 어떤 값을 비교하나요?
3. NMS 예제에서 A와 B의 IoU는 약 0.667입니다. IoU Threshold 0.5와 0.75에서 왜 다른 결과가 나왔나요?
4. **오탐지(FP)·미탐지(FN)라고 판단하려면** 무엇이 추가로 필요할까요? (정답 Annotation)

**다음:** `day4_03_sam_segmentation_v1.ipynb`에서는 같은 사진의 특정 객체를 SAM 2.1로 분할하고 Prompt별 Mask를 비교합니다. ⑦교시에서는 이 Notebook이 저장한 `outputs/yolo/yolo_detections.json`을 다시 사용합니다.